<a href="https://colab.research.google.com/github/philipmikh/CS1ReviewForCS2/blob/master/VERSION_6_0.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ============================================================
# Mic -> Whisper -> (Sentence-by-sentence) Emotion Prediction
# NOW USING k-NN (trained on centroid vectors from emotion_avg.pkl)
# Streams results: every time a sentence completes, it gets labeled.
# ============================================================

# --- Install deps (Colab) ---
!apt-get -y update
!apt-get -y install ffmpeg
!pip -q install gradio faster-whisper soundfile torch sentence-transformers scikit-learn numpy pandas

import os
import re
import pickle
import tempfile
import numpy as np
import pandas as pd
import gradio as gr
import soundfile as sf

from faster_whisper import WhisperModel
from sentence_transformers import SentenceTransformer
from sklearn.neighbors import KNeighborsClassifier

# -----------------------------
# 0) Load centroids from Drive
# -----------------------------
from google.colab import drive
drive.mount("/content/drive")

CENTROIDS_PATH = "/content/drive/MyDrive/CLASS/emotion_project/emotion_avg.pkl"

if not os.path.exists(CENTROIDS_PATH):
    raise FileNotFoundError(
        f"Could not find {CENTROIDS_PATH}. Upload emotion_avg.pkl or fix the path."
    )

with open(CENTROIDS_PATH, "rb") as f:
    emotion_avg = pickle.load(f)

for k in list(emotion_avg.keys()):
    emotion_avg[k] = np.array(emotion_avg[k])

EMOTIONS = list(emotion_avg.keys())
print("Loaded emotions:", EMOTIONS)
print("Count:", len(EMOTIONS))

# -----------------------------
# 1) Pick device (GPU if available)
# -----------------------------
def pick_device():
    try:
        import torch
        return "cuda" if torch.cuda.is_available() else "cpu"
    except:
        return "cpu"

device = pick_device()
compute_type = "float16" if device == "cuda" else "int8"
print("Using device:", device)

# -----------------------------
# 2) Load Whisper model
# -----------------------------
whisper = WhisperModel(
    "base",
    device=device,
    compute_type=compute_type
)

# -----------------------------
# 3) Load embedder (AUTO-MATCH centroid dimensions)
# -----------------------------
any_emotion = next(iter(emotion_avg.keys()))
centroid_dim = int(np.array(emotion_avg[any_emotion]).shape[-1])
print("Centroid dim:", centroid_dim)

if centroid_dim == 384:
    EMBED_MODEL_NAME = "all-MiniLM-L6-v2"
elif centroid_dim == 768:
    EMBED_MODEL_NAME = "all-mpnet-base-v2"
else:
    raise ValueError(
        f"Unknown centroid dim {centroid_dim}. Tell me what model you used to build emotion_avg.pkl."
    )

print("Using embed model:", EMBED_MODEL_NAME)
embedder = SentenceTransformer(EMBED_MODEL_NAME)

# -----------------------------
# 3.5) Build + train k-NN on centroids
# -----------------------------
# One centroid per emotion => k must be 1 (true k-NN with k>1 requires many samples/class)
X_train = np.vstack([emotion_avg[e].reshape(1, -1) for e in EMOTIONS])  # shape: (num_emotions, dim)
y_train = np.array(EMOTIONS)

knn = KNeighborsClassifier(
    n_neighbors=1,
    metric="cosine",     # cosine distance = 1 - cosine_similarity
    algorithm="brute",
    weights="distance"
)
knn.fit(X_train, y_train)
print("k-NN trained on centroids. k=1, metric=cosine")

# -----------------------------
# 4) Audio input -> filepath (Gradio-safe)
# -----------------------------
def audio_to_path(audio_input):
    """
    Handles:
    - filepath (str)
    - dict with "path"
    - tuple (sample_rate, numpy_array)
    """
    if audio_input is None:
        return None

    if isinstance(audio_input, str):
        return audio_input

    if isinstance(audio_input, dict) and "path" in audio_input:
        return audio_input["path"]

    if isinstance(audio_input, (tuple, list)) and len(audio_input) == 2:
        sr, data = audio_input
        tmp = tempfile.NamedTemporaryFile(suffix=".wav", delete=False)
        sf.write(tmp.name, data, int(sr))
        return tmp.name

    return None

# -----------------------------
# 5) Emotion prediction for ONE sentence (k-NN)
# -----------------------------
def predict_emotion_sentence_knn(sentence: str):
    sentence = (sentence or "").strip()
    if not sentence:
        return "unclear", 0.0

    emb = embedder.encode([sentence])[0].reshape(1, -1)

    # Get two nearest centroids to compute confidence (top1_sim - top2_sim)
    n2 = 2 if len(EMOTIONS) >= 2 else 1
    dists, idxs = knn.kneighbors(emb, n_neighbors=n2, return_distance=True)

    # Convert cosine distance -> cosine similarity
    sim1 = 1.0 - float(dists[0][0])
    pred = str(knn.predict(emb)[0])

    if n2 == 2:
        sim2 = 1.0 - float(dists[0][1])
    else:
        sim2 = -1.0

    confidence = float(sim1 - sim2)
    return pred, confidence

# -----------------------------
# 6) Sentence splitter (robust enough for spoken text)
# -----------------------------
SENT_END_RE = re.compile(r"(.+?[.!?]+)(\s+|$)", re.DOTALL)

def extract_complete_sentences(buffer_text: str):
    """
    Returns (sentences, remainder_buffer)
    sentences = list of full sentences ending in . ! ?
    """
    sentences = []
    idx = 0
    for m in SENT_END_RE.finditer(buffer_text):
        sent = m.group(1).strip()
        if sent:
            sentences.append(sent)
        idx = m.end()
    remainder = buffer_text[idx:].strip()
    return sentences, remainder

# -----------------------------
# 7) Main pipeline (STREAMING)
#    Audio -> Whisper segments -> whenever a full sentence forms -> classify + yield UI updates
# -----------------------------
CONF_THRESHOLD = 0.05  # raise to be stricter; lower to label more often

def transcribe_stream_and_label(audio_input):
    audio_path = audio_to_path(audio_input)
    if audio_path is None or not os.path.exists(audio_path):
        yield "No audio detected.", "unclear", 0.0, pd.DataFrame(columns=["sentence", "emotion", "confidence"])
        return

    try:
        segments, info = whisper.transcribe(audio_path, beam_size=5)
    except Exception as e:
        yield f"Transcription error: {e}", "unclear", 0.0, pd.DataFrame(columns=["sentence", "emotion", "confidence"])
        return

    running_transcript = ""
    sentence_buffer = ""
    rows = []
    last_emotion = "unclear"
    last_conf = 0.0

    for seg in segments:
        chunk = (seg.text or "").strip()
        if not chunk:
            continue

        running_transcript = (running_transcript + " " + chunk).strip()
        sentence_buffer = (sentence_buffer + " " + chunk).strip()

        complete_sents, sentence_buffer = extract_complete_sentences(sentence_buffer)

        for s in complete_sents:
            emo, conf = predict_emotion_sentence_knn(s)
            final_emo = emo if conf >= CONF_THRESHOLD else "unclear"

            rows.append({"sentence": s, "emotion": final_emo, "confidence": conf})
            last_emotion = final_emo
            last_conf = conf

            df = pd.DataFrame(rows)
            yield running_transcript, last_emotion, last_conf, df

    df = pd.DataFrame(rows)
    yield running_transcript, last_emotion, last_conf, df

# -----------------------------
# 8) Gradio UI
# -----------------------------
with gr.Blocks() as demo:
    gr.Markdown("## 🎤 Whisper → Sentence-by-sentence Emotion Prediction (k-NN)")

    audio = gr.Audio(sources=["microphone"], type="filepath", label="Record your voice")
    btn = gr.Button("Transcribe + Label Sentences")

    transcript_out = gr.Textbox(label="Running Transcript", lines=6)
    last_emotion_out = gr.Textbox(label="Latest Sentence Emotion", lines=1)
    last_conf_out = gr.Number(label="Latest Confidence (top1 - top2)")
    table_out = gr.Dataframe(label="Per-sentence results", interactive=False)

    btn.click(
        transcribe_stream_and_label,
        inputs=audio,
        outputs=[transcript_out, last_emotion_out, last_conf_out, table_out]
    )

demo.launch(debug=True, share=True)

Get:1 https://cli.github.com/packages stable InRelease [3,917 B]
Get:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:3 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Hit:4 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:5 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:6 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:7 https://cli.github.com/packages stable/main amd64 Packages [357 B]
Get:8 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [85.2 kB]
Get:9 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Get:10 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Get:11 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease [24.6 kB]
Get:12 https://r2u.stat.illinois.edu/ubuntu jammy/main all Packages [9,825 kB]
Get:13 https://r2u.stat.illinois.edu/ubuntu jammy/main amd64 Packages [2,926 kB]
Get:14 http://

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Centroid dim: 768
Using embed model: all-mpnet-base-v2


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

k-NN trained on centroids. k=1, metric=cosine
Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://aaa93698b82d87eeff.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
